# BO7 Aimbot V5 - YOLOv11s 640x640 Training
## Massiv bessere Erkennung durch:
- **YOLOv11s** statt YOLOv8n (bessere Architektur)
- **640x640** statt 320x320 (4x mehr Pixel = viel bessere Erkennung)
- **Groessere Datensaetze** von Roboflow (BO7 + Warzone)
- **100 Epochen** Training

### Anleitung:
1. Runtime > Change runtime type > **T4 GPU**
2. Alle Zellen nacheinander ausfuehren
3. Am Ende das `.onnx` Modell herunterladen

In [ ]:
# Schritt 1: GPU pruefen und Pakete installieren
!nvidia-smi
!pip install ultralytics roboflow --quiet

In [ ]:
# Schritt 2: Google Drive verbinden (zum Speichern)
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/BO7_V5_Training'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Speicherort: {SAVE_DIR}')

In [ ]:
# Schritt 3: Datensaetze von Roboflow herunterladen
# OPTION A: BO7 Datensatz (938 Bilder - Enemy Klasse)
# OPTION B: Warzone Datensatz (5895 Bilder - groesster verfuegbarer)
# OPTION C: BO6 Datensatz (1675 Bilder - Player Klasse)
#
# Du brauchst einen kostenlosen Roboflow Account:
# 1. Geh auf https://app.roboflow.com/ und registriere dich
# 2. Geh auf https://app.roboflow.com/settings/api und kopiere deinen API Key
# 3. Fuege ihn unten ein

ROBOFLOW_API_KEY = "DEIN_API_KEY_HIER"  # <-- HIER EINFUEGEN!

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# BO7 Datensatz herunterladen (938 Bilder)
print('Lade BO7 Datensatz...')
try:
    project = rf.workspace('models-7p3lx').project('bo7-du6xg')
    dataset_bo7 = project.version(1).download('yolov11', location='/content/dataset_bo7')
    print(f'BO7 Datensatz geladen!')
except Exception as e:
    print(f'BO7 Download fehlgeschlagen: {e}')
    print('Versuche alternativen Datensatz...')
    # Fallback: BO6 Datensatz (1675 Bilder)
    project = rf.workspace('bo6-xvjgj').project('bo6-enemy-detection')
    dataset_bo7 = project.version(1).download('yolov11', location='/content/dataset_bo7')
    print(f'BO6 Datensatz geladen (als Fallback)!')

In [ ]:
# Schritt 4: Datensatz pruefen
import glob

# Finde die data.yaml Datei
yaml_files = glob.glob('/content/dataset_bo7/**/data.yaml', recursive=True)
if not yaml_files:
    yaml_files = glob.glob('/content/dataset_bo7/**/*.yaml', recursive=True)

if yaml_files:
    DATA_YAML = yaml_files[0]
    print(f'data.yaml gefunden: {DATA_YAML}')
    with open(DATA_YAML, 'r') as f:
        print(f.read())
else:
    print('FEHLER: Keine data.yaml gefunden!')
    print('Verfuegbare Dateien:')
    for f in glob.glob('/content/dataset_bo7/**/*', recursive=True)[:20]:
        print(f'  {f}')

# Bilder zaehlen
train_imgs = glob.glob('/content/dataset_bo7/**/train/images/*', recursive=True)
val_imgs = glob.glob('/content/dataset_bo7/**/valid/images/*', recursive=True)
print(f'\nTraining Bilder: {len(train_imgs)}')
print(f'Validation Bilder: {len(val_imgs)}')

In [ ]:
# Schritt 5: YOLOv11s Training starten!
# v11s = Small Variante (besser als Nano, schnell genug fuer Echtzeit)
# 640x640 = 4x mehr Pixel als unser altes 320x320 Modell
# 100 Epochen = gruendliches Training

from ultralytics import YOLO

# YOLOv11 Small laden (vortrainiert auf COCO)
model = YOLO('yolo11s.pt')

# Training starten
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,           # T4 GPU vertraegt 16 bei 640px
    patience=20,        # Frueh stoppen wenn kein Fortschritt
    save=True,
    project=SAVE_DIR,
    name='bo7_v5',
    exist_ok=True,
    # Augmentation fuer bessere Generalisierung
    flipud=0.0,         # Kein vertikales Flip (Spieler stehen aufrecht)
    fliplr=0.5,         # Horizontales Flip
    mosaic=1.0,         # Mosaic Augmentation
    mixup=0.1,          # Leichtes Mixup
    degrees=5.0,        # Leichte Rotation
    translate=0.1,      # Leichte Verschiebung
    scale=0.3,          # Skalierung
    hsv_h=0.01,         # Minimale Farbvariation
    hsv_s=0.3,          # Saettigung
    hsv_v=0.3,          # Helligkeit
)

print('\n=== TRAINING FERTIG ===')
print(f'Ergebnisse in: {SAVE_DIR}/bo7_v5/')

In [ ]:
# Schritt 6: Ergebnisse anzeigen
import glob
from IPython.display import Image, display

# Finde results.png
results_imgs = glob.glob(f'{SAVE_DIR}/bo7_v5/results.png')
if results_imgs:
    display(Image(results_imgs[0], width=800))

# Zeige mAP Werte
import csv
csv_files = glob.glob(f'{SAVE_DIR}/bo7_v5/results.csv')
if csv_files:
    with open(csv_files[0], 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        if rows:
            last = rows[-1]
            print(f"\n=== FINALE METRIKEN ===")
            for key in last:
                if 'map' in key.lower() or 'precision' in key.lower() or 'recall' in key.lower():
                    print(f"  {key.strip()}: {last[key].strip()}")

In [ ]:
# Schritt 7: Export zu ONNX (640x640, dynamische Dimensionen)
import glob

# Finde best.pt
best_pt = glob.glob(f'{SAVE_DIR}/bo7_v5/weights/best.pt')
if not best_pt:
    best_pt = glob.glob(f'{SAVE_DIR}/bo7_v5*/weights/best.pt')

if best_pt:
    model = YOLO(best_pt[0])
    model.export(
        format='onnx',
        imgsz=640,
        dynamic=True,
        simplify=True,
    )
    print(f'\nONNX Modell exportiert!')
    
    # Kopiere ONNX zu Google Drive mit klarem Namen
    import shutil
    onnx_src = best_pt[0].replace('.pt', '.onnx')
    onnx_dst = f'{SAVE_DIR}/bo7_v5_640.onnx'
    shutil.copy2(onnx_src, onnx_dst)
    print(f'Gespeichert als: {onnx_dst}')
    print(f'\n=== FERTIG! Lade diese Datei herunter: ===')
    print(f'{onnx_dst}')
else:
    print('FEHLER: best.pt nicht gefunden!')

In [ ]:
# Schritt 8: Modell herunterladen
from google.colab import files

onnx_path = f'{SAVE_DIR}/bo7_v5_640.onnx'
if os.path.exists(onnx_path):
    files.download(onnx_path)
    print('Download gestartet!')
else:
    print(f'Datei nicht gefunden: {onnx_path}')
    print('Manuell herunterladen aus Google Drive!')